# ML Coding — Day 4: MLP / MoE I

**~1 hour, vectorized NumPy.** The feed-forward half of a transformer block, with **forward paired
with backward** (gradients checked by finite differences via `_numgrad`), then sparse mixture-of-experts
routing. No per-element loops in solutions (a loop over the handful of experts is the algorithm and is
allowed). Run the **helpers cell first**.

**Q1** GELU fwd+bwd `[warmup]` · **Q2** MLP block fwd+bwd `[medium]` · **Q3** SwiGLU fwd+bwd
`[medium]` · **Q4** top-k MoE routing `[hard]` · **Q5** MoE via grouped einsum `[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np

_C = np.sqrt(2.0 / np.pi)


def _softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def _gelu(x):                                          # tanh approximation (GPT-style)
    x = np.asarray(x, float)
    return 0.5 * x * (1.0 + np.tanh(_C * (x + 0.044715 * x ** 3)))


def _silu(x):
    x = np.asarray(x, float)
    return x / (1.0 + np.exp(-x))


def _numgrad(f, x, eps=1e-6):
    x = np.asarray(x, float)
    grad = np.zeros_like(x)
    flat, gflat = x.reshape(-1), grad.reshape(-1)
    for i in range(flat.size):
        o = flat[i]
        flat[i] = o + eps; a = f(x)
        flat[i] = o - eps; b = f(x)
        flat[i] = o
        gflat[i] = (a - b) / (2 * eps)
    return grad

## Q1 — GELU forward + backward  ·  `[warmup]`

**Paper:** Hendrycks & Gimpel, *Gaussian Error Linear Units*, 2016 (arXiv:1606.08415). **Why it
matters:** GELU is the default transformer activation; here we use the **tanh approximation** used by
GPT (pure NumPy, smooth, finite-diff-checkable).

Implement `gelu(x)` and `gelu_backward(g, x)` (elementwise; `g` is the upstream gradient):
`gelu(x) = 0.5·x·(1 + tanh(u))`, `u = √(2/π)·(x + 0.044715·x³)`. Derive `gelu'(x)` and return `g·gelu'(x)`.

In [ ]:
import numpy as np


def gelu(x):
    raise NotImplementedError


def gelu_backward(g, x):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    rng = np.random.default_rng(0)
    x = rng.standard_normal(40)
    assert np.allclose(gelu(x), _gelu(x))
    g = rng.standard_normal(40)
    dx = gelu_backward(g, x)
    assert np.allclose(dx, _numgrad(lambda z: np.sum(g * gelu(z)), x), atol=1e-6)
    print("Q1 OK")

_q1()

## Q2 — MLP block forward + backward  ·  `[medium]`

**Why it matters:** the transformer FFN is `Linear → GELU → Linear`, and its backward is the canonical
multi-layer backprop you must be able to write from memory.

`x (N,Din)`. Implement:
- `mlp_forward(x, W1, b1, W2, b2) -> y`  with `h = gelu(x@W1 + b1)`, `y = h@W2 + b2`.
- `mlp_backward(dy, x, W1, b1, W2, b2) -> (dx, dW1, db1, dW2, db2)`.

**Hint:** `dW2 = hᵀ·dy`, `db2 = Σ dy`, `dh = dy·W2ᵀ`, `dz1 = dh ⊙ gelu'(z1)`, `dW1 = xᵀ·dz1`,
`db1 = Σ dz1`, `dx = dz1·W1ᵀ`. No loops.

In [ ]:
def mlp_forward(x, W1, b1, W2, b2):
    raise NotImplementedError


def mlp_backward(dy, x, W1, b1, W2, b2):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(1)
    N, Din, H, Dout = 4, 5, 7, 3
    x = rng.standard_normal((N, Din))
    W1 = rng.standard_normal((Din, H)); b1 = rng.standard_normal(H)
    W2 = rng.standard_normal((H, Dout)); b2 = rng.standard_normal(Dout)
    y = mlp_forward(x, W1, b1, W2, b2)
    assert y.shape == (N, Dout)
    g = rng.standard_normal((N, Dout))
    dx, dW1, db1, dW2, db2 = mlp_backward(g, x, W1, b1, W2, b2)
    L = lambda *a: np.sum(g * mlp_forward(*a))
    assert np.allclose(dx, _numgrad(lambda v: L(v, W1, b1, W2, b2), x), atol=1e-5)
    assert np.allclose(dW1, _numgrad(lambda v: L(x, v, b1, W2, b2), W1), atol=1e-5)
    assert np.allclose(db1, _numgrad(lambda v: L(x, W1, v, W2, b2), b1), atol=1e-5)
    assert np.allclose(dW2, _numgrad(lambda v: L(x, W1, b1, v, b2), W2), atol=1e-5)
    assert np.allclose(db2, _numgrad(lambda v: L(x, W1, b1, W2, v), b2), atol=1e-5)
    print("Q2 OK")

_q2()

## Q3 — SwiGLU forward + backward  ·  `[medium]`

**Paper:** Shazeer, *GLU Variants Improve Transformer*, 2020 (arXiv:2002.05202). **Why it matters:**
SwiGLU is the FFN in LLaMA/PaLM — a **gated** unit that consistently beats plain GELU-MLP. It has two
input projections multiplied together, so the backward has a product rule.

`x (N,Din)`, `Wg, Wu (Din,H)`, `Wd (H,Dout)`. Implement:
- `swiglu_forward(x, Wg, Wu, Wd) -> y`  with `g = silu(x@Wg)`, `u = x@Wu`, `p = g ⊙ u`, `y = p@Wd`.
  (`silu(z) = z·σ(z)`.)
- `swiglu_backward(dy, x, Wg, Wu, Wd) -> (dx, dWg, dWu, dWd)`.

**Hint:** `dWd = pᵀ·dy`, `dp = dy·Wdᵀ`, `dg = dp ⊙ u`, `du = dp ⊙ g`, `da = dg ⊙ silu'(x@Wg)`;
then `dWg = xᵀ·da`, `dWu = xᵀ·du`, `dx = da·Wgᵀ + du·Wuᵀ`. No loops.

In [ ]:
def swiglu_forward(x, Wg, Wu, Wd):
    raise NotImplementedError


def swiglu_backward(dy, x, Wg, Wu, Wd):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(2)
    N, Din, H, Dout = 4, 5, 6, 3
    x = rng.standard_normal((N, Din))
    Wg = rng.standard_normal((Din, H)); Wu = rng.standard_normal((Din, H)); Wd = rng.standard_normal((H, Dout))
    y = swiglu_forward(x, Wg, Wu, Wd)
    assert y.shape == (N, Dout)
    g = rng.standard_normal((N, Dout))
    dx, dWg, dWu, dWd = swiglu_backward(g, x, Wg, Wu, Wd)
    L = lambda *a: np.sum(g * swiglu_forward(*a))
    assert np.allclose(dx, _numgrad(lambda v: L(v, Wg, Wu, Wd), x), atol=1e-5)
    assert np.allclose(dWg, _numgrad(lambda v: L(x, v, Wu, Wd), Wg), atol=1e-5)
    assert np.allclose(dWu, _numgrad(lambda v: L(x, Wg, v, Wd), Wu), atol=1e-5)
    assert np.allclose(dWd, _numgrad(lambda v: L(x, Wg, Wu, v), Wd), atol=1e-5)
    print("Q3 OK")

_q3()

## Q4 — Top-k MoE routing (forward)  ·  `[hard]`

**Papers:** Shazeer et al., *Sparsely-Gated MoE*, 2017 (arXiv:1701.06538); Fedus et al., *Switch
Transformer*, 2021 (arXiv:2101.03961). **Why it matters:** conditional compute — each token is routed
to only its top-`k` of `E` experts, so capacity grows without proportional FLOPs. The crux is the
**dispatch/combine**: gather each token to its experts, run them, scatter the weighted results back.

Experts are stacked: `W1 (E,Din,H)`, `b1 (E,H)`, `W2 (E,H,Din)`, `b2 (E,Din)`; gate `Wg (Din,E)`.
Implement `moe_forward(X, Wg, W1, b1, W2, b2, k) -> Y (N,Din)`: `gate = softmax(X@Wg)`; pick top-`k`
experts per token; **renormalize** their gate probs to sum to 1; each expert is `gelu(·@W1+b1)@W2+b2`;
`Y[n] = Σ_{e∈topk(n)} w_{n,e} · expert_e(X[n])`. (Loop over the `E` experts is fine; vectorize over the
tokens routed to each.)

In [ ]:
def moe_forward(X, Wg, W1, b1, W2, b2, k):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _moe_oracle(X, Wg, W1, b1, W2, b2, k):
    gate = _softmax(X @ Wg)
    N, D = X.shape
    idx = np.argsort(-gate, axis=1)[:, :k]
    out = np.zeros((N, D))
    for n in range(N):
        ws = gate[n, idx[n]]; ws = ws / ws.sum()
        for j, e in enumerate(idx[n]):
            out[n] += ws[j] * (_gelu(X[n] @ W1[e] + b1[e]) @ W2[e] + b2[e])
    return out


def _q4():
    rng = np.random.default_rng(3)
    N, D, H, E, k = 6, 4, 5, 3, 2
    X = rng.standard_normal((N, D)); Wg = rng.standard_normal((D, E))
    W1 = rng.standard_normal((E, D, H)); b1 = rng.standard_normal((E, H))
    W2 = rng.standard_normal((E, H, D)); b2 = rng.standard_normal((E, D))
    out = moe_forward(X, Wg, W1, b1, W2, b2, k)
    assert out.shape == (N, D)
    assert np.allclose(out, _moe_oracle(X, Wg, W1, b1, W2, b2, k), atol=1e-10)
    print("Q4 OK")

_q4()

## Q5 — MoE via grouped einsum  ·  `[hard · tensor trick]`

**Papers:** GShard (Lepikhin et al. 2020, arXiv:2006.16668); Switch Transformer (Fedus 2021). **Why it
matters:** real MoE kernels avoid a Python loop over experts by doing **one grouped (batched) matmul**
over the expert axis. **The trick:** run *all* experts on *all* tokens with `np.einsum` over the expert
dimension, then combine with the sparse routing-weight matrix — no per-expert loop.

Same signature: `moe_grouped(X, Wg, W1, b1, W2, b2, k) -> Y`, identical result to Q4. Build the
`(N,E)` weight matrix `Wsel` (renormalized top-`k` gate probs, 0 elsewhere), then
`H1 = einsum('nd,edh->enh', X, W1) + b1`, `A = gelu(H1)`, `Yexp = einsum('enh,ehd->end', A, W2) + b2`,
`Y = einsum('ne,end->nd', Wsel, Yexp)`. (Dense — computes every expert; note the FLOPs trade-off vs the
true sparse dispatch.) No per-element or per-expert loops.

In [ ]:
def moe_grouped(X, Wg, W1, b1, W2, b2, k):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(4)
    N, D, H, E, k = 7, 4, 6, 4, 2
    X = rng.standard_normal((N, D)); Wg = rng.standard_normal((D, E))
    W1 = rng.standard_normal((E, D, H)); b1 = rng.standard_normal((E, H))
    W2 = rng.standard_normal((E, H, D)); b2 = rng.standard_normal((E, D))
    out = moe_grouped(X, Wg, W1, b1, W2, b2, k)
    assert out.shape == (N, D)
    assert np.allclose(out, _moe_oracle(X, Wg, W1, b1, W2, b2, k), atol=1e-10)
    print("Q5 OK")

_q5()

---
## Reference solutions (don't peek until you've tried)

*Backward of the MoE block is left as a follow-up — it adds router-gradient bookkeeping on top of the
per-expert MLP backward from Q2.*

In [ ]:
import numpy as np

_C = np.sqrt(2.0 / np.pi)


def _g(x):
    x = np.asarray(x, float)
    return 0.5 * x * (1.0 + np.tanh(_C * (x + 0.044715 * x ** 3)))


def _gp(x):                                            # gelu'(x)
    x = np.asarray(x, float)
    u = _C * (x + 0.044715 * x ** 3)
    t = np.tanh(u)
    du = _C * (1.0 + 3 * 0.044715 * x ** 2)
    return 0.5 * (1.0 + t) + 0.5 * x * (1.0 - t ** 2) * du


def _sig(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, float)))


def _sm(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def ref_gelu(x):
    return _g(x)


def ref_gelu_backward(g, x):
    return g * _gp(x)


def ref_mlp_forward(x, W1, b1, W2, b2):
    x = np.asarray(x, float)
    h = _g(x @ W1 + b1)
    return h @ W2 + b2


def ref_mlp_backward(dy, x, W1, b1, W2, b2):
    x = np.asarray(x, float); dy = np.asarray(dy, float)
    z1 = x @ W1 + b1
    h = _g(z1)
    dW2 = h.T @ dy
    db2 = dy.sum(axis=0)
    dh = dy @ W2.T
    dz1 = dh * _gp(z1)
    dW1 = x.T @ dz1
    db1 = dz1.sum(axis=0)
    dx = dz1 @ W1.T
    return dx, dW1, db1, dW2, db2


def ref_swiglu_forward(x, Wg, Wu, Wd):
    x = np.asarray(x, float)
    g = x @ Wg * _sig(x @ Wg)                          # silu(x@Wg)
    u = x @ Wu
    return (g * u) @ Wd


def ref_swiglu_backward(dy, x, Wg, Wu, Wd):
    x = np.asarray(x, float); dy = np.asarray(dy, float)
    a = x @ Wg
    sa = _sig(a)
    g = a * sa                                         # silu(a)
    u = x @ Wu
    p = g * u
    dWd = p.T @ dy
    dp = dy @ Wd.T
    dg = dp * u
    du = dp * g
    da = dg * (sa * (1.0 + a * (1.0 - sa)))            # silu'(a)
    dWg = x.T @ da
    dWu = x.T @ du
    dx = da @ Wg.T + du @ Wu.T
    return dx, dWg, dWu, dWd


def _route(X, Wg, k):
    gate = _sm(X @ Wg)
    N, E = gate.shape
    idx = np.argpartition(-gate, k - 1, axis=1)[:, :k]
    rows = np.arange(N)[:, None]
    w = gate[rows, idx]
    w = w / w.sum(axis=1, keepdims=True)
    Wsel = np.zeros((N, E))
    Wsel[rows, idx] = w
    return Wsel


def ref_moe_forward(X, Wg, W1, b1, W2, b2, k):
    X = np.asarray(X, float)
    Wsel = _route(X, Wg, k)
    N, D = X.shape
    E = Wg.shape[1]
    out = np.zeros((N, D))
    for e in range(E):                                 # loop over experts (the algorithm)
        m = Wsel[:, e] > 0
        if not m.any():
            continue
        Xe = X[m]
        ye = _g(Xe @ W1[e] + b1[e]) @ W2[e] + b2[e]
        out[m] += Wsel[m, e][:, None] * ye
    return out


def ref_moe_grouped(X, Wg, W1, b1, W2, b2, k):
    X = np.asarray(X, float)
    Wsel = _route(X, Wg, k)
    H1 = np.einsum("nd,edh->enh", X, W1) + b1[:, None, :]
    A = _g(H1)
    Yexp = np.einsum("enh,ehd->end", A, W2) + b2[:, None, :]
    return np.einsum("ne,end->nd", Wsel, Yexp)


_x = np.linspace(-3, 3, 7)
assert np.allclose(ref_gelu(_x), _g(_x))
rng = np.random.default_rng(11)
_X = rng.standard_normal((5, 4)); _Wg = rng.standard_normal((4, 3))
_W1 = rng.standard_normal((3, 4, 6)); _b1 = rng.standard_normal((3, 6))
_W2 = rng.standard_normal((3, 6, 4)); _b2 = rng.standard_normal((3, 4))
assert np.allclose(ref_moe_forward(_X, _Wg, _W1, _b1, _W2, _b2, 2),
                   ref_moe_grouped(_X, _Wg, _W1, _b1, _W2, _b2, 2))
print("reference solutions OK")